# 1. Simulate EHT 2017 observations of an orbiting hot spot

This notebook builds the dataset for the Fourier-domain (black-hole imaging) tutorial:

1. **Synthesize a movie** of an *m-ring plus orbiting hot spot* — a bright compact emission
   region circling a Sgr A*-like ring, the standard test case for dynamical black-hole imaging.
2. **Observe it with the April 2017 EHT array** (8 stations) using
   [ehtim](https://github.com/achael/eht-imaging), producing the sparse visibility products
   NeuralDMD trains on: per-frame forward operators, complex visibilities, amplitudes, and
   closure phases.

**Environment**: this notebook needs `ehtim` (and `astropy`, `scikit-image`) but **not** JAX —
run it in your ehtim environment. Notebooks 02–04 need JAX but not ehtim.

The exact format of every output file, and all observation-generation details, are documented in
[`eht2017/README.md`](../../eht2017/README.md).

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("../../eht2017")  # observation pipeline + EHT2017 array file

from make_movie import make_frames, to_ehtim_movie, save_movie_hdf5, SGRA
from data_generation import Config, generate

data_root = "./data"
os.makedirs(data_root, exist_ok=True)

## Synthesize the m-ring + hot spot movie

The scene has two components:

- a **thick ring** (radius 23 μas, FWHM 25 μas) with a mild `m = 1` azimuthal brightness
  asymmetry (an *m-ring*), carrying ~2.5 Jy;
- a **Gaussian hot spot** (FWHM 28 μas, ~0.28 Jy — about 10% of the total flux) orbiting at
  radius 25.6 μas with an **80-minute period** (counterclockwise on-sky).

We generate 411 frames spanning a 6-hour observation (9h–15h UT), i.e. **4.5 full orbits**,
on a 200×200 grid with a 200 μas field of view at Sgr A*'s sky coordinates.

In [ ]:
num_frames = 411
tstart_hr, tstop_hr = 9.0, 15.0
npix, fov_uas = 200, 200.0

movie_params = dict(
    # static m-ring
    r_ring=23.0,          # ring radius [uas]
    width_fwhm=25.0,      # ring thickness [uas]
    beta1=0.12 * np.exp(1j * np.deg2rad(35.0)),  # m=1 azimuthal asymmetry
    flux=2.47,            # ring flux [Jy]
    # orbiting hot spot
    r_orbit=25.6,         # orbital radius [uas]
    period_min=80.0,      # orbital period [minutes]
    phase0_deg=178.0,     # phase at t = tstart
    direction=-1,         # sense of rotation in array coords (-1 = CCW on-sky)
    spot_fwhm=28.0,       # spot size [uas]
    spot_flux=0.28,       # spot flux [Jy]
)

frames, times = make_frames(
    num_frames=num_frames, tstart_hr=tstart_hr, tstop_hr=tstop_hr,
    npix=npix, fov_uas=fov_uas, **movie_params,
)
print("frames:", frames.shape, " flux/frame: %.3f Jy" % frames[0].sum())

In [ ]:
# a few frames across the first orbit
idxs = np.linspace(0, int(num_frames * (movie_params["period_min"] / 60.0) / (tstop_hr - tstart_hr)), 6).astype(int)
fig, axes = plt.subplots(1, len(idxs), figsize=(3 * len(idxs), 3))
for ax, i in zip(axes, idxs):
    ax.imshow(frames[i], cmap="afmhot")
    ax.set_title(f"t = {times[i]:.2f} h", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Save as an ehtim movie

`to_ehtim_movie` wraps the frames into an `ehtim.movie.Movie` at Sgr A*'s coordinates
(RA 17.76 h, Dec −29.0°, 227 GHz, MJD 57854 — the 2017 campaign) and we save it in ehtim's
hdf5 format, which the observation pipeline reads back.

In [ ]:
movie = to_ehtim_movie(frames, times, fov_uas=fov_uas, **SGRA)
movie_path = os.path.join(data_root, "mring+hs.hdf5")
save_movie_hdf5(movie, movie_path)

## Observe with the EHT 2017 array

`generate` simulates the full observation (see [`eht2017/README.md`](../../eht2017/README.md)
for details):

- schedules scans matched to the movie's frame cadence (~53 s), with 5 s integrations;
- downsamples the movie ×4 to the **50×50 model grid** (flux-preserving) and computes each
  frame's visibilities with a direct Fourier transform;
- adds thermal noise from the station SEFDs plus 4% fractional systematic noise;
- extracts complex visibilities, amplitudes, and closure phases, pads them to fixed shapes,
  and writes the `.npy` products + diagnostics.

Each frame is seen by at most **42 visibilities** (≤ 35 closure triangles) — severely
underconstrained per frame, which is exactly the regime NeuralDMD targets.

⏱ Expect roughly **15–30 minutes** for 411 frames.

In [ ]:
cfg = Config(
    movie_name="mring+hs",
    movie_dir=data_root,
    movie_file="mring+hs.hdf5",
    output_root=data_root,
    fractional_noise=0.04,
    scale_factor=4,
)
obs_dir = generate(cfg)
print("\nDataset written to:", obs_dir)

## Inspect the dataset

Sanity checks: the (u,v) coverage accumulated over the night, the number of visibilities per
frame (stations rise and set), and the visibility amplitude versus baseline length. The
`diagnostics.json` χ² values compare the noisy data against the ground-truth movie itself —
they are reduced per degree of freedom, and land *below* 1 (≈ 0.6 for the visibilities)
because `add_fractional_noise` inflates the error budget σ without perturbing the data.
Treat them as the effective noise floor for training.

In [ ]:
import json

uv = np.load(os.path.join(obs_dir, "uv_coords.npy"))     # (T, M, 2) in Glambda
amp = np.load(os.path.join(obs_dir, "amp_targets.npy"))
masks = np.load(os.path.join(obs_dir, "masks.npy"))
num_vis = np.load(os.path.join(obs_dir, "num_vis_list.npy"))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

valid = masks > 0
axes[0].scatter(uv[..., 0][valid], uv[..., 1][valid], s=1, alpha=0.3)
axes[0].scatter(-uv[..., 0][valid], -uv[..., 1][valid], s=1, alpha=0.3, color="C0")
axes[0].set_xlabel(r"u [G$\lambda$]"); axes[0].set_ylabel(r"v [G$\lambda$]")
axes[0].set_title("EHT 2017 (u,v) coverage, full night")
axes[0].set_aspect("equal")

axes[1].plot(num_vis)
axes[1].set_xlabel("frame"); axes[1].set_ylabel("# visibilities")
axes[1].set_title("Visibilities per frame")

uvdist = np.hypot(uv[..., 0], uv[..., 1])[valid]
axes[2].semilogy(uvdist, amp[valid], ".", ms=2, alpha=0.3)
axes[2].set_xlabel(r"|uv| [G$\lambda$]"); axes[2].set_ylabel("amplitude [Jy]")
axes[2].set_title("Amplitude vs baseline length")

plt.tight_layout(); plt.show()

print(json.dumps(json.load(open(os.path.join(obs_dir, "diagnostics.json"))), indent=2))

**Next:** [`02_pretrain_disk_template.ipynb`](02_pretrain_disk_template.ipynb) — initialize
NeuralDMD's spatial modes with a Zernike disk template (switch to your JAX environment).